# phase_2 detector on Kaggle — YOLOv8l, Drive-resumable

Trains the 29-region detector **exactly like the local run**, but checkpoints to
**Google Drive via rclone** so it survives Kaggle's ephemeral storage.

**Why Drive:** `/kaggle/working` is wiped between sessions unless you *Commit*, and
a session can die mid-run. We push the run dir to Drive **every `SYNC_EVERY`
optimizer steps + each epoch**; the next session pulls `last.pt` and `--resume`s.

**Each session:** Settings → Accelerator → **GPU**, then *Run All*. From the 2nd
session on, set `RESUME = 1` in CONFIG. That's the only change.

## CONFIG — edit these

In [1]:
import os
# ===== code comes from GitHub (cloned in the next cell), not a code dataset =====
PHASE2_SRC = "/kaggle/working/repo/phase_2"
# ===== Kaggle dataset slugs (edit) =====
IMAGES      = "/kaggle/input/datasets/nguynnghin/mimic-cxr-448/mimic-cxr-448"  # 448 images (stem == image_id)
YOLO_LABELS = "/kaggle/input/datasets/nguynnghin/yolo-labels"   # PREBUILT labels (gold 784 ids skipped at link time)

# ===== training (OPTION A: fast warm-start run that fits one session) =====
# MODEL doubles as the INIT-WEIGHTS knob: keep "yolov8m.pt" for a fresh run, OR set it to a
# saved best.pt path to CONTINUE training from an earlier run (warm start, RESUME stays 0).
MODEL    = "yolov8m.pt"   # m (not l): ~2x faster, near-identical box quality for 29 easy regions
IMGSZ    = 448            # images are 448 (bump to 640 later if tiny regions like carina/svc drag mAP)
EPOCHS   = 30             # 29 regions converge fast; patience 15 cuts earlier if mAP plateaus
FRACTION = 0.25           # ~37k imgs: full data + 50ep only reached ~0.68, so data is NOT the lever
RUN_NAME = "det29"
RESUME   = 0             # 0 = fresh; 1 = continue an INTERRUPTED run from the Drive checkpoint

# ===== durable checkpoints (Google Drive via a SERVICE ACCOUNT) =====
# dhint: is defined entirely by the SA + the folder id below -> no rclone.conf, no token.
# Because root = your CHEX-DATA folder, paths DROP the 'CHEX-DATA/' prefix.
DRIVE_FOLDER_ID = "1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O"   # CHEX-DATA folder id
RCLONE_REMOTE   = "dhint:phase2_runs"           # = CHEX-DATA/phase2_runs
SYNC_EVERY      = 300                            # push every N optimizer steps

for k, v in dict(PHASE2_SRC=PHASE2_SRC, IMAGES=IMAGES, YOLO_LABELS=YOLO_LABELS, MODEL=MODEL,
                 IMGSZ=IMGSZ, EPOCHS=EPOCHS, FRACTION=FRACTION, RUN_NAME=RUN_NAME, RESUME=RESUME,
                 DRIVE_FOLDER_ID=DRIVE_FOLDER_ID, RCLONE_REMOTE=RCLONE_REMOTE, SYNC_EVERY=SYNC_EVERY).items():
    os.environ[k] = str(v)
print("config set | MODEL =", MODEL, "| FRACTION =", FRACTION, "| EPOCHS =", EPOCHS, "| RESUME =", RESUME)

config set | MODEL = yolov8m.pt | FRACTION = 0.25 | EPOCHS = 30 | RESUME = 0


In [2]:
# --- get the code from GitHub (instead of uploading a code dataset). Internet ON. ---
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!ls /kaggle/working/repo/phase_2 | head

build_pseudo_scene_graph.py
build_sft_dataset.py
build_yolo_dataset.py
config.py
constants.py
eval_yolo.py
extract_sg_vocab.py
finetune_sg_llm.py
gold_ids.txt
infer_yolo.py


## 1 — install rclone + auth via a Google **service account**

No OAuth token, no `rclone.conf`. **One-time setup:**
1. **Add-ons → Secrets → Add**: label **`GDRIVE_SA`**, value = the **full JSON** of your
   service-account file (the whole `{...}`).
2. In Google Drive, **share the `CHEX-DATA` folder** (Editor) with the SA's email
   (`client_email` in the JSON, e.g. `kaggle-rclone@…iam.gserviceaccount.com`) — a service
   account has its **own empty Drive**, so it can't see your folder until you share it.
3. Put the `CHEX-DATA` **folder id** (URL `…/folders/<ID>`) into `DRIVE_FOLDER_ID` in CONFIG.

In [3]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working
  curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip
  cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

rclone v1.74.3


In [4]:
import os
# Drive remote 'dhint' via YOUR OAuth token (NOT a service account: SA has no storage quota
# and 403s on upload to My Drive). Token comes from `rclone authorize "drive"` -> Kaggle
# Secret GDRIVE_TOKEN. Graceful: if missing/unwritable, training still runs WITHOUT sync.
SYNC_OK = "0"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GDRIVE_TOKEN").strip()   # the {...} from rclone authorize
    os.environ["RCLONE_CONFIG_DHINT_TYPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_TOKEN"] = token
    os.environ["RCLONE_CONFIG_DHINT_SCOPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID"] = os.environ["DRIVE_FOLDER_ID"]  # = CHEX-DATA (yours)
    os.environ.pop("RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE", None)  # bỏ SA cũ còn sót trong kernel
    remote = os.environ["RCLONE_REMOTE"]
    if os.system('rclone mkdir "%s"' % remote) == 0 and \
       os.system('echo ok | rclone rcat "%s/_write_test.txt"' % remote) == 0:
        # SA failed exactly at the upload step -> prove we can WRITE before training
        SYNC_OK = "1"
        os.system('rclone delete "%s/_write_test.txt" 2>/dev/null' % remote)
        print("remote OK (OAuth, write verified) ->", remote)
        os.system('rclone lsd dhint: 2>/dev/null | head')
    else:
        print("[WARN] Drive write FAILED -> check GDRIVE_TOKEN (own account, write scope) + DRIVE_FOLDER_ID")
except Exception as e:
    print("[WARN] GDRIVE_TOKEN secret not set -> NO Drive sync:", e)
os.environ["SYNC_OK"] = SYNC_OK
print("SYNC_OK =", SYNC_OK)

2026/06/28 05:04:27 NOTICE: Config file "/root/.config/rclone/rclone.conf" not found - using defaults
2026/06/28 05:04:28 NOTICE: Config file "/root/.config/rclone/rclone.conf" not found - using defaults


remote OK (OAuth, write verified) -> dhint:phase2_runs
           0 2025-12-04 06:49:35        -1 CheXplus
           0 2025-12-04 17:00:28        -1 Mimic-CXR
           0 2026-06-19 02:33:01        -1 TRACE
           0 2026-05-02 03:57:01        -1 chexplus_processed
           0 2026-06-19 02:33:19        -1 mimic_processed
           0 2026-06-26 17:19:57        -1 phase2_runs
SYNC_OK = 1


## 2 — copy code into the writable dir + install ultralytics

In [5]:
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!pip install -q ultralytics

/kaggle/working/phase_2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 65.0 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx

In [6]:
# verify GPU BEFORE training (P100 = sm_60 is NOT supported by torch 2.10 -> use T4!)
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
assert torch.cuda.is_available(), "No GPU! Settings -> Accelerator -> GPU T4 x2, then Run All."
assert torch.cuda.get_device_capability(0) >= (7, 0), \
    "GPU too old for torch 2.10 (need sm_70+). P100 is sm_60 -> switch to T4."

torch: 2.10.0+cu128 | cuda: True | device_count: 2
gpu: Tesla T4 | capability: (7, 5)


## 3 — assemble the YOLO dataset (labels built LOCALLY)

The slow part (open every image for W/H + convert boxes) was done **once locally** with
`build_yolo_dataset.py --labels-only` and uploaded as the **`yolo-labels`** dataset. Here we just
**symlink the matching images** + write `dataset.yaml` — fast, no PIL, no path autodetect headaches.
So you no longer upload the 11 GB scene-graph dataset or the metadata to Kaggle at all.

In [7]:
import os

# Trỏ trực tiếp thư mục nhãn vào biến môi trường YOLO_LABELS
LDIR = os.environ["YOLO_LABELS"]
os.environ["LDIR"] = LDIR

# Chạy script tạo dataset
!python link_yolo_images.py --labels-dir "$LDIR" --images-root "$IMAGES" --out /kaggle/working/yolo_ds

exclude ids : 784 held-out stems (from /kaggle/working/phase_2/gold_ids.txt)
labels root : /kaggle/input/datasets/nguynnghin/yolo-labels/labels/labels
images root : /kaggle/input/datasets/nguynnghin/mimic-cxr-448/mimic-cxr-448
Indexing images (one walk, no PIL) ...
  images indexed: 377,105

[DONE] linked per split: {'train': 148242, 'val': 21335, 'test': 42194}
[HELD OUT] gold/excluded skipped per split: {'train': 0, 'val': 0, 'test': 784} (784 total — never linked into images/labels)
dataset.yaml -> /kaggle/working/yolo_ds/dataset.yaml


In [8]:
# sanity: dataset.yaml + how many images/labels linked per split (gold 784 skipped from test)
import os, glob
DS = "/kaggle/working/yolo_ds"
print(open(f"{DS}/dataset.yaml").read())
for s in ("train", "val", "test"):
    imgs = glob.glob(f"{DS}/images/{s}/*")
    lbls = glob.glob(f"{DS}/labels/{s}/*")
    print(f"{s}: images={len(imgs)} labels={len(lbls)}")
sample = glob.glob(f"{DS}/images/train/*")[:1]
if sample:
    print("symlink ->", os.path.realpath(sample[0]))

path: /kaggle/working/yolo_ds
train: images/train
val: images/val
test: images/test
nc: 29
names:
  0: abdomen
  1: aortic arch
  2: cardiac silhouette
  3: carina
  4: cavoatrial junction
  5: left apical zone
  6: left clavicle
  7: left costophrenic angle
  8: left hemidiaphragm
  9: left hilar structures
  10: left lower lung zone
  11: left lung
  12: left mid lung zone
  13: left upper lung zone
  14: mediastinum
  15: right apical zone
  16: right atrium
  17: right clavicle
  18: right costophrenic angle
  19: right hemidiaphragm
  20: right hilar structures
  21: right lower lung zone
  22: right lung
  23: right mid lung zone
  24: right upper lung zone
  25: spine
  26: svc
  27: trachea
  28: upper mediastinum

train: images=148242 labels=148242
val: images=21335 labels=21335
test: images=42194 labels=42194
symlink -> /kaggle/input/datasets/nguynnghin/mimic-cxr-448/mimic-cxr-448/p14/p14398566/MIMIC_p14398566_s59460454_b3db4b5b-2749f445-10482b3a-b679ada9-0eea6994.jpg


## 4 — train (Drive-synced)

Fresh run pushes checkpoints to Drive every `SYNC_EVERY` steps + each epoch.
If the session dies, set `RESUME=1` in CONFIG and *Run All* again — `train_yolo.py`
pulls `last.pt` from Drive and continues.

In [9]:
# background Drive sync (DDP-safe) — RUN THIS BEFORE the train cell.
# ultralytics DDP (--device 0,1) does NOT propagate model.add_callback hooks to its worker
# processes, so kaggle_sync's callbacks won't fire under 2 GPUs. This daemon thread is
# INDEPENDENT of them: it mirrors /kaggle/working/runs -> Drive every SYNC_SECONDS. It keeps
# running while the next cell's `!python train_yolo.py` blocks (threads run during the
# subprocess wait), so checkpoints reach Drive mid-run even with DDP.
import os, threading, subprocess

SYNC_SECONDS = 300
_bg_stop = threading.Event()

def _bg_sync():
    remote = os.environ["RCLONE_REMOTE"]          # dhint:phase2_runs
    runs = "/kaggle/working/runs"                  # contains det29/ -> lands at phase2_runs/det29
    while not _bg_stop.wait(SYNC_SECONDS):
        if os.path.isdir(runs):
            try:
                subprocess.run(["rclone", "copy", runs, remote,
                                "--transfers", "8", "--checkers", "8", "--quiet"],
                               check=False, timeout=600)
                print(f"[bg-sync] pushed runs/ -> {remote}", flush=True)
            except Exception as e:
                print(f"[bg-sync] push failed (continuing): {e}", flush=True)

if os.environ.get("SYNC_OK") == "1":
    threading.Thread(target=_bg_sync, daemon=True).start()
    print(f"[bg-sync] started: runs/ -> {os.environ['RCLONE_REMOTE']} every {SYNC_SECONDS}s")
else:
    print("[bg-sync] Drive off (SYNC_OK != 1) -> no background sync")

[bg-sync] started: runs/ -> dhint:phase2_runs every 300s


In [10]:
import os
# 2-GPU DDP. Mid-run Drive sync is done by the BACKGROUND thread above (DDP drops the
# kaggle_sync callbacks). We still pass --sync-remote so a --resume can PULL last.pt from
# Drive first, but set --sync-every huge so the (non-firing) callback never double-pushes
# against the background thread. --batch is FIXED (autobatch -1 is unreliable under DDP);
# 32 total = 16 per T4.
SR = os.environ["RCLONE_REMOTE"] if os.environ.get("SYNC_OK") == "1" else ""
os.environ["SYNC"] = ("--sync-remote %s --sync-every 99999999" % SR) if SR else ""
os.environ["RESUME_FLAG"] = "--resume" if int(os.environ["RESUME"]) else ""
print("2-GPU DDP | bg-sync handles Drive | resume:", os.environ["RESUME_FLAG"] or "(fresh)")
!python train_yolo.py \
  --data /kaggle/working/yolo_ds/dataset.yaml \
  --model "$MODEL" --device 0,1 --imgsz $IMGSZ --epochs $EPOCHS --fraction $FRACTION --batch 32 \
  --runs /kaggle/working/runs --name "$RUN_NAME" $SYNC $RESUME_FLAG

2-GPU DDP | bg-sync handles Drive | resume: (fresh)
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[sync] rclone sync ON -> dhint:phase2_runs  (every 99999999 steps + each epoch)
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/yolo_ds/dataset.yaml, degrees=2.0, deterministic=True, device=0,1, dfl=1.5, dis=

2026/06/28 05:40:39 ERROR : det29/weights: error reading destination directory: couldn't list directory: googleapi: Error 403: Quota exceeded for quota metric 'Queries' and limit 'Queries per minute' of service 'drive.googleapis.com' for consumer 'project_number:202264815644'.
Details:
[
  {
    "@type": "type.googleapis.com/google.rpc.ErrorInfo",
    "domain": "googleapis.com",
    "metadata": {
      "consumer": "projects/202264815644",
      "quota_limit": "defaultPerMinutePerProject",
      "quota_limit_value": "840000",
      "quota_location": "global",
      "quota_metric": "drive.googleapis.com/default",
      "quota_unit": "1/min/{project}",
      "service": "drive.googleapis.com"
    },
    "reason": "RATE_LIMIT_EXCEEDED"
  },
  {
    "@type": "type.googleapis.com/google.rpc.Help",
    "links": [
      {
        "description": "Request a higher quota limit.",
        "url": "https://cloud.google.com/docs/quotas/help/request_increase"
      }
    ]
  }
]
, rateLimitExceeded


       3/30      4.05G      1.114     0.6287      1.332        451        448: 56% ━━━━━━╸───── 660/1159 2.6it/s 4:20<3:13

2026/06/28 05:41:45 ERROR : Attempt 1/3 failed with 1 errors and: couldn't list directory: googleapi: Error 403: Quota exceeded for quota metric 'Queries' and limit 'Queries per minute' of service 'drive.googleapis.com' for consumer 'project_number:202264815644'.
Details:
[
  {
    "@type": "type.googleapis.com/google.rpc.ErrorInfo",
    "domain": "googleapis.com",
    "metadata": {
      "consumer": "projects/202264815644",
      "quota_limit": "defaultPerMinutePerProject",
      "quota_limit_value": "840000",
      "quota_location": "global",
      "quota_metric": "drive.googleapis.com/default",
      "quota_unit": "1/min/{project}",
      "service": "drive.googleapis.com"
    },
    "reason": "RATE_LIMIT_EXCEEDED"
  },
  {
    "@type": "type.googleapis.com/google.rpc.Help",
    "links": [
      {
        "description": "Request a higher quota limit.",
        "url": "https://cloud.google.com/docs/quotas/help/request_increase"
      }
    ]
  }
]
, rateLimitExceeded


       3/30      4.05G      1.114     0.6287      1.332        439        448: 57% ━━━━━━╸───── 665/1159 2.8it/s 4:22<2:54[bg-sync] pushed runs/ -> dhint:phase2_runs


...

      19/30      4.06G     0.8967     0.4634      1.197        434        448: 1% ──────────── 17/1159 2.3it/s 7.4s<8:09

2026/06/28 08:08:43 ERROR : Attempt 1/3 failed with 1 errors and: Post "https://www.googleapis.com/upload/drive/v3/files/1o7trODnmJEvzgMRmLPUf0Jm92mc-AhRj?alt=json&fields=id%2Cname%2Csize%2Cmd5Checksum%2Csha1Checksum%2Csha256Checksum%2Ctrashed%2CexplicitlyTrashed%2CmodifiedTime%2CcreatedTime%2CmimeType%2Cparents%2CwebViewLink%2CshortcutDetails%2CexportLinks%2CresourceKey&setModifiedDate=true&supportsAllDrives=true&uploadType=resumable&upload_id=AJ5rDhEIf7Fcbqtaji8SPfTIFRs3KYANt2kEA7Fy_3O4rSbAFCfSQDoBKr2KUkbnhT-VIlFnIP_F2oB9MrJhUIN34IO9zkPT9M6jgKejeMyeg6E&session_crd=AHSoBRUahwxU8QQltlDKNHxw1nlhIJg-ybLgrke65rGi84jMxkb04VumJcMN1co4lIHsavqfcjp8PXM1Bqc": can't copy - source file is being updated (size changed from 155589745 to 155589937)


      19/30      4.06G     0.8967     0.4672      1.195        442        448: 3% ──────────── 42/1159 2.5it/s 17.6s<7:19[bg-sync] pushed runs/ -> dhint:phase2_runs
      19/30      4.06G     0.8932     0.4648      1.192        452        448: 3% ──────────── 43/1159 2.8it/s 17.9s<6:41

2026/06/28 08:08:54 ERROR : Attempt 2/3 succeeded


      19/30      4.06G     0.8769     0.4571      1.191        423        448: 70% ━━━━━━━━──── 812/1159 2.5it/s 5:20<2:178[bg-sync] pushed runs/ -> dhint:phase2_runs
      19/30      4.06G     0.8791     0.4584      1.193         54        448: 100% ━━━━━━━━━━━━ 1159/1159 2.5it/s 7:360.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 334/334 3.2it/s 1:430.3sss
                   all      21335     606919      0.933      0.887       0.93       0.69

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      20/30      4.06G     0.8651     0.4504      1.182        414        448: 37% ━━━━──────── 430/1159 2.9it/s 2:50<4:143[bg-sync] pushed runs/ -> dhint:phase2_runs
      20/30      4.06G     0.8673     0.4505      1.182         58        448: 100% ━━━━━━━━━━━━ 1159/1159 2.5it/s 7:350.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 17% ━━────────── 59/334

## 5 — eval mAP (val; use `--split test` for the gold human-verified set)

In [11]:
!python eval_yolo.py \
  --weights /kaggle/working/runs/$RUN_NAME/weights/best.pt \
  --data /kaggle/working/yolo_ds/dataset.yaml --split val --device 0

Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 93 layers, 25,856,551 parameters, 0 gradients, 78.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 83.8±36.5 MB/s, size: 44.9 KB)
val: Scanning /kaggle/working/yolo_ds/labels/val.cache... 21335 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 21335/21335 2.7Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 48% ━━━━━╸────── 645/1334 4.0it/s 2:49<2:546[bg-sync] pushed runs/ -> dhint:phase2_runs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1334/1334 3.9it/s 5:430.3ss
                   all      21335     606919      0.934      0.889      0.931      0.694
               abdomen      21266      21266      0.944      0.978       0.99      0.854
           aortic arch      21065      21065      0.961      0.895      0.933      0.649
    cardiac silhouette      213

## 6 — grab the trained model

`best.pt` is already on Drive (final push at train end). To pull it anywhere:

```bash
rclone copy dhint:phase2_runs/det29/weights/best.pt ./   # dhint: root = CHEX-DATA
```
Or download it from the Kaggle notebook's **Output** tab (`runs/det29/weights/best.pt`).

In [12]:
import os
RUN = os.environ["RUN_NAME"]
if os.environ.get("SYNC_OK") == "1":
    os.system('rclone copy /kaggle/working/runs/%s "%s/%s" --quiet' % (RUN, os.environ["RCLONE_REMOTE"], RUN))
    print("pushed final run -> %s/%s" % (os.environ["RCLONE_REMOTE"], RUN))
else:
    print("Drive sync off -> download from the notebook 'Output' tab: runs/%s/weights/best.pt" % RUN)
!ls -lh /kaggle/working/runs/$RUN_NAME/weights/

pushed final run -> dhint:phase2_runs/det29
total 990M
-rw-r--r-- 1 root root  50M Jun 28 10:02 best.pt
-rw-r--r-- 1 root root 149M Jun 28 05:27 epoch0.pt
-rw-r--r-- 1 root root 149M Jun 28 07:02 epoch10.pt
-rw-r--r-- 1 root root 149M Jun 28 07:49 epoch15.pt
-rw-r--r-- 1 root root 149M Jun 28 08:37 epoch20.pt
-rw-r--r-- 1 root root 149M Jun 28 09:24 epoch25.pt
-rw-r--r-- 1 root root 149M Jun 28 06:15 epoch5.pt
-rw-r--r-- 1 root root  50M Jun 28 10:02 last.pt
